# PHASE 2 — Phương sai seed thật

**Mục tiêu:** Huấn luyện lại cấu hình canonical với seed 42, 43, 44 để đo MDE.

**Cấu hình canonical:**
- Config: `configs/sage_b2_crack500.yaml` (B2, Dual S2-Gate, SoftBIoU λ=0.50, 35 epochs, two-stage)
- Dataset: Crack500_ready (train/val/test split chuẩn)
- Eval: `predict_full_image_direct`, tau=0.5

**Quy tắc cứng:**
1. KHÔNG xem test. Chỉ báo Val Dice.
2. Mỗi seed = 1 thư mục riêng `results/final_protocol/phase2_seed{S}/`
3. Lưu `config_snapshot.yaml`, `git_hash.txt`, `sha256_checkpoint.txt`, seed vào thư mục.
4. Seed 42 đã chạy (Run2) nhưng KHÔNG phải multi-seed có chủ đích → phải train lại sạch.

In [ ]:
# ==========================================
# CELL 1: Mount Drive & Clone Repo
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

import os

# Thay bằng path thực tế trên Drive của bạn
DRIVE_ROOT = '/content/drive/MyDrive/SAGE_LITE'  # <-- SỬA NẾU CẦN
DATASET_ROOT = '/content/drive/MyDrive/Crack500_ready'  # <-- SỬA NẾU CẦN

# Clone repo hoặc dùng code từ Drive
# Option A: clone GitHub
# !git clone https://github.com/bach0823/SAGE_LITE /content/SAGE_LITE

# Option B: dùng code từ Drive (nhanh hơn)
SAGE_LITE_CODE = '/content/drive/MyDrive/SAGE_LITE_code'  # <-- SỬA NẾU CẦN
import sys
sys.path.insert(0, SAGE_LITE_CODE)
sys.path.insert(0, os.path.join(SAGE_LITE_CODE, 'scripts'))
os.chdir(SAGE_LITE_CODE)

print('Drive mounted. CWD:', os.getcwd())

In [ ]:
# ==========================================
# CELL 2: Cài dependencies
# ==========================================
!pip install -q timm albumentations scipy scikit-learn

In [ ]:
# ==========================================
# CELL 3: Kiểm tra GPU và Git hash
# ==========================================
import torch, subprocess, hashlib, shutil
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('CUDA:', torch.version.cuda)

try:
    git_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=SAGE_LITE_CODE).decode().strip()
    print('Git hash:', git_hash)
except:
    git_hash = 'unknown'
    print('Git hash: unknown (not a git repo)')

In [ ]:
# ==========================================
# CELL 4: Hàm tiện ích
# ==========================================
import json, yaml

PHASE2_OUT = '/content/drive/MyDrive/phase2_results'  # <-- SỬA NẾU CẦN
os.makedirs(PHASE2_OUT, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest().upper()

def save_run_metadata(run_dir, seed, git_hash, checkpoint_path, val_result):
    """Lưu metadata đầy đủ cho mỗi run."""
    with open(os.path.join(run_dir, 'git_hash.txt'), 'w') as f:
        f.write(git_hash)
    sha = sha256_file(checkpoint_path)
    with open(os.path.join(run_dir, 'sha256_checkpoint.txt'), 'w') as f:
        f.write(sha)
    meta = {
        'seed': seed,
        'git_hash': git_hash,
        'checkpoint_sha256': sha,
        'val_result': val_result
    }
    with open(os.path.join(run_dir, 'run_metadata.json'), 'w') as f:
        json.dump(meta, f, indent=2)
    print(f'[Metadata saved] seed={seed}, sha256={sha[:16]}..., val_dice={val_result.get("dice","?")}')

print('Utilities ready.')

In [ ]:
# ==========================================
# CELL 5: Hàm đánh giá Val (không dùng test)
# ==========================================
import cv2
import numpy as np
from evaluate_crack_official import get_image_mask_pairs, predict_full_image_direct, compute_boundary_iou
from sage.utils.advanced_metrics import calculate_hd95_bf1

def eval_val_direct(model, config_dict, device, tau=0.5, diagnostic=True):
    """Đánh giá trên val, KHÔNG bao giờ chạm test."""
    pairs = get_image_mask_pairs(config_dict, 'val')
    print(f'Val pairs: {len(pairs)}')
    all_dice, all_prec, all_rec, all_biou, all_hd95 = [], [], [], [], []
    total_tp = total_fp = total_fn = 0
    for img_p, mask_p in pairs:
        img = cv2.cvtColor(cv2.imread(img_p), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_p, cv2.IMREAD_GRAYSCALE)
        target = (mask > 127).astype(np.uint8)
        logits = predict_full_image_direct(model, img, device, target_size=448)
        prob = 1.0 / (1.0 + np.exp(-logits))
        pred = (prob >= tau).astype(np.uint8)
        if pred.shape != target.shape:
            pred = cv2.resize(pred.astype(np.float32),
                              (target.shape[1], target.shape[0]),
                              interpolation=cv2.INTER_NEAREST).astype(np.uint8)
        tp = int(np.sum((pred==1)&(target==1)))
        fp = int(np.sum((pred==1)&(target==0)))
        fn = int(np.sum((pred==0)&(target==1)))
        total_tp += tp; total_fp += fp; total_fn += fn
        prec = tp/(tp+fp+1e-5); rec = tp/(tp+fn+1e-5)
        dice = 2*tp/(2*tp+fp+fn+1e-5)
        all_dice.append(dice); all_prec.append(prec); all_rec.append(rec)
        if diagnostic:
            all_biou.append(compute_boundary_iou(pred, target, d=2))
            hd, _ = calculate_hd95_bf1(pred, target)
            all_hd95.append(hd)
    result = {
        'dice': float(np.mean(all_dice)),
        'precision': float(np.mean(all_prec)),
        'recall': float(np.mean(all_rec)),
        'global_pixel_iou': float(total_tp/(total_tp+total_fp+total_fn+1e-5)),
        'tau': tau,
        'n_images': len(pairs)
    }
    if diagnostic:
        result['boundary_iou'] = float(np.mean(all_biou)) if all_biou else 0.0
        result['hd95'] = float(np.mean(all_hd95)) if all_hd95 else 999.0
    return result

print('Eval function ready.')

In [ ]:
# ==========================================
# CELL 6: Train một seed
# Chạy cell này 3 lần: seed=42, 43, 44
# THAY seed ở đây mỗi lần
# ==========================================
SEED = 42  # <-- ĐỔI THÀNH 43 hoặc 44 cho run tiếp theo

run_name = f'phase2_seed{SEED}'
run_dir  = os.path.join(PHASE2_OUT, run_name)
os.makedirs(run_dir, exist_ok=True)

# Config snapshot
config_path = os.path.join(SAGE_LITE_CODE, 'configs', 'sage_b2_crack500.yaml')
shutil.copy(config_path, os.path.join(run_dir, 'config_snapshot.yaml'))

print(f'=== Training seed={SEED} ===')
print(f'Output: {run_dir}')

# Train command
!python scripts/train_crack.py \
    --config configs/sage_b2_crack500.yaml \
    --seed {SEED} \
    --data-root {DATASET_ROOT} \
    --output-dir {run_dir} \
    --num-workers 2 \
    2>&1 | tee {run_dir}/train.log

print(f'Training done for seed={SEED}')

In [ ]:
# ==========================================
# CELL 7: Đánh giá Val sau training
# Chạy ngay sau CELL 6 cho cùng seed
# ==========================================
import yaml
from sage.networks import create_b2_unet

config_path = os.path.join(run_dir, 'config_snapshot.yaml')
with open(config_path) as f:
    cfg = yaml.safe_load(f)
cfg['root_dir'] = DATASET_ROOT

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sage_cfg = cfg.get('sage_config', {})
model = create_b2_unet(
    num_transformer_layers=int(cfg.get('num_transformer_layers', 4)),
    pretrained=False, sage_config=sage_cfg,
    p3_mode=cfg.get('p3_mode'),
    use_plu_head=cfg.get('use_plu_head', False),
    use_s2_gate=cfg.get('use_s2_gate', False),
    s2_gate_kernel_size=cfg.get('s2_gate_kernel_size', 3),
    use_s2_gate_block2=cfg.get('use_s2_gate_block2', False),
    s2_gate_block2_kernel_size=cfg.get('s2_gate_block2_kernel_size', 3),
    s2_gate_block2_dilation=cfg.get('s2_gate_block2_dilation', 1),
).to(device)

# Dùng best_model_b2_global.pth (best stage 2 global)
ckpt_path = os.path.join(run_dir, 'best_model_b2_global.pth')
ckpt = torch.load(ckpt_path, map_location=device)
if 'model_state_dict' in ckpt:
    model.load_state_dict(ckpt['model_state_dict'])
else:
    model.load_state_dict(ckpt)
model.eval()
print(f'Loaded: {ckpt_path}')
if 'best_dice' in ckpt: print(f'  best_dice in meta: {ckpt["best_dice"]:.6f}')

val_result = eval_val_direct(model, cfg, device, tau=0.5, diagnostic=True)
print('Val result:', val_result)

save_run_metadata(run_dir, SEED, git_hash, ckpt_path, val_result)
print(f'=== seed={SEED} done: Val Dice = {val_result["dice"]:.6f} ===')

In [ ]:
# ==========================================
# CELL 8: Tổng hợp 3 seed — chạy SAU KHI có đủ seed 42, 43, 44
# ==========================================
import json
import numpy as np

seeds = [42, 43, 44]
dices, precisions, recalls = [], [], []
rows = []

print('=== Phase 2 Summary ===')
print(f'{"Seed":>6} {"Val Dice":>10} {"Precision":>10} {"Recall":>10} {"BIoU":>8} {"HD95":>8}')

for s in seeds:
    meta_path = os.path.join(PHASE2_OUT, f'phase2_seed{s}', 'run_metadata.json')
    if not os.path.exists(meta_path):
        print(f'seed {s}: CHƯA CHẠY')
        continue
    with open(meta_path) as f:
        m = json.load(f)
    r = m['val_result']
    dices.append(r['dice'])
    precisions.append(r['precision'])
    recalls.append(r['recall'])
    rows.append(r)
    print(f'{s:>6} {r["dice"]:>10.6f} {r["precision"]:>10.6f} {r["recall"]:>10.6f} '
          f'{r.get("boundary_iou",0):>8.4f} {r.get("hd95",0):>8.2f}')

if len(dices) >= 2:
    mean_d = np.mean(dices); std_d = np.std(dices)
    mde = 2 * std_d
    print(f'\nMean Val Dice: {mean_d:.6f}')
    print(f'Std  Val Dice: {std_d:.6f}')
    print(f'Min/Max:       {min(dices):.6f} / {max(dices):.6f}')
    print(f'MDE (2×std):   {mde:.6f}  ← ngưỡng phát hiện tối thiểu cho Phase 3+')
    
    summary = {
        'seeds': seeds[:len(dices)],
        'val_dices': dices,
        'mean_dice': float(mean_d),
        'std_dice': float(std_d),
        'min_dice': float(min(dices)),
        'max_dice': float(max(dices)),
        'mde': float(mde),
        'per_seed': rows
    }
    out = os.path.join(PHASE2_OUT, 'phase2_summary.json')
    with open(out, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f'Saved: {out}')
    
    # Kiểm tra TTA flip_hv từ Phase 1 có vượt MDE không
    tta_delta = 0.003158  # từ Phase 1
    print(f'\nPhase 1 TTA delta: {tta_delta:.6f}')
    print(f'MDE:               {mde:.6f}')
    if tta_delta > mde:
        print('→ TTA flip_hv VƯỢT MDE → xác nhận có nghĩa thực tế')
    else:
        print('→ TTA flip_hv KHÔNG vượt MDE → chưa có bằng chứng vượt noise seed')
else:
    print('Chưa đủ seed để tổng hợp.')

In [ ]:
# ==========================================
# CELL 9 (TÙY CHỌN): Ensemble 3 seed
# Chạy sau khi có đủ 3 checkpoints
# ==========================================
from sage.networks import create_b2_unet

models_ensemble = []
for s in seeds:
    ckpt_path = os.path.join(PHASE2_OUT, f'phase2_seed{s}', 'best_model_b2_global.pth')
    if not os.path.exists(ckpt_path):
        print(f'Missing: {ckpt_path}')
        continue
    m = create_b2_unet(
        num_transformer_layers=4, pretrained=False,
        sage_config=cfg.get('sage_config', {}),
        use_s2_gate=True, s2_gate_kernel_size=3,
        use_s2_gate_block2=True, s2_gate_block2_kernel_size=3, s2_gate_block2_dilation=2
    ).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    if 'model_state_dict' in ckpt:
        m.load_state_dict(ckpt['model_state_dict'])
    else:
        m.load_state_dict(ckpt)
    m.eval()
    models_ensemble.append(m)

print(f'{len(models_ensemble)} models loaded for ensemble.')

if len(models_ensemble) == 3:
    # Ensemble: average prob maps
    pairs = get_image_mask_pairs(cfg, 'val')
    all_dice = []
    for img_p, mask_p in pairs:
        img = cv2.cvtColor(cv2.imread(img_p), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_p, cv2.IMREAD_GRAYSCALE)
        target = (mask > 127).astype(np.uint8)
        probs = []
        for m in models_ensemble:
            logits = predict_full_image_direct(m, img, device, target_size=448)
            probs.append(1.0 / (1.0 + np.exp(-logits)))
        prob_avg = np.mean(probs, axis=0)
        pred = (prob_avg >= 0.5).astype(np.uint8)
        if pred.shape != target.shape:
            pred = cv2.resize(pred.astype(np.float32),
                              (target.shape[1], target.shape[0]),
                              interpolation=cv2.INTER_NEAREST).astype(np.uint8)
        tp = int(np.sum((pred==1)&(target==1)))
        fp = int(np.sum((pred==1)&(target==0)))
        fn = int(np.sum((pred==0)&(target==1)))
        all_dice.append(2*tp/(2*tp+fp+fn+1e-5))
    ensemble_dice = float(np.mean(all_dice))
    print(f'\nEnsemble (3 seeds) Val Dice: {ensemble_dice:.6f}')
    print(f'vs Mean single-seed:          {np.mean(dices):.6f}')
    print(f'Delta ensemble vs mean:        {ensemble_dice - np.mean(dices):.6f}')